# PatchCamelyon Experiment Visualization

This notebook visualizes PCam experiment results including dataset exploration, training curves, model performance metrics, and prediction analysis.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path

# Import PCam dataset
import sys
sys.path.insert(0, str(Path('../../src/data').resolve()))
from pcam_dataset import PCamDataset

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 8)

# Output directory
OUTPUT_DIR = Path('../../results/pcam')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded successfully.')

## 1. Dataset Exploration

Load and display sample images from the PCam dataset along with class distribution.

In [ ]:
# Configuration
DATA_ROOT = Path('../../data/pcam')
SAMPLE_SIZE = 64  # 8x8 grid

# Load a subset of training data for visualization
try:
    dataset = PCamDataset(
        root_dir=str(DATA_ROOT),
        split='train',
        download=False
    )
    
    # Sample indices
    total_samples = len(dataset)
    sample_indices = np.random.choice(total_samples, size=min(SAMPLE_SIZE, total_samples), replace=False)
    
    # Collect samples
    samples = []
    labels = []
    for idx in sample_indices:
        sample = dataset[idx]
        samples.append(sample['image'])
        labels.append(sample['label'].item())
    
    samples = torch.stack(samples)
    labels = np.array(labels)
    
    print(f'Loaded {len(samples)} samples from PCam dataset')
    print(f'Class distribution: Normal={np.sum(labels==0)}, Metastatic={np.sum(labels==1)}')
    
    DATASET_LOADED = True
except Exception as e:
    print(f'Could not load dataset: {e}')
    DATASET_LOADED = False

In [ ]:
# Display 8x8 grid of sample images
if DATASET_LOADED:
    fig, axes = plt.subplots(8, 8, figsize=(16, 16))
    fig.suptitle('PCam Sample Images (8x8 Grid)', fontsize=16, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        if i < len(samples):
            img = samples[i]
            # Denormalize for display if needed (images are [0,1] after ToTensor)
            if img.max() <= 1.0:
                img = img.numpy().transpose(1, 2, 0)
            else:
                img = img.numpy().transpose(1, 2, 0) / 255.0
            
            ax.imshow(img)
            ax.set_title(f'L={labels[i]}', fontsize=8)
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sample_grid.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved sample grid to {OUTPUT_DIR / "sample_grid.png"}')

In [ ]:
# Plot class distribution bar chart
if DATASET_LOADED:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    class_names = ['Normal (0)', 'Metastatic (1)']
    class_counts = [np.sum(labels == 0), np.sum(labels == 1)]
    
    colors = ['#2ecc71', '#e74c3c']
    bars = ax.bar(class_names, class_counts, color=colors, edgecolor='black', linewidth=1.2)
    
    ax.set_xlabel('Class', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('PCam Class Distribution', fontsize=14, fontweight='bold')
    
    # Add count labels on bars
    for bar, count in zip(bars, class_counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{count}', ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved class distribution to {OUTPUT_DIR / "class_distribution.png"}')

## 2. Training Curves

Load metrics from checkpoint or JSON and plot loss and accuracy curves over epochs.

In [ ]:
# Load metrics from checkpoint or JSON
def load_metrics(checkpoint_path=None, json_path=None):
    """Load training metrics from checkpoint or JSON file."""
    metrics = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    # Try multiple possible paths
    possible_checkpoint_paths = [
        checkpoint_path,
        Path('../../checkpoints/pcam/best_model.pth'),
        Path('../../checkpoints/pcam/checkpoint_epoch_5.pth')
    ]
    
    for path in possible_checkpoint_paths:
        if path and Path(path).exists():
            try:
                checkpoint = torch.load(path, map_location='cpu')
                if 'history' in checkpoint:
                    metrics = checkpoint['history']
                    print(f'Loaded metrics from checkpoint: {path}')
                    return metrics
                elif 'metrics' in checkpoint:
                    # Single epoch metrics - create history
                    metrics = {
                        'train_loss': [checkpoint['metrics'].get('train_loss', 0)],
                        'val_loss': [checkpoint['metrics'].get('val_loss', 0)],
                        'train_acc': [checkpoint['metrics'].get('train_accuracy', 0)],
                        'val_acc': [checkpoint['metrics'].get('val_accuracy', 0)]
                    }
                    print(f'Loaded single epoch metrics from checkpoint: {path}')
                    return metrics
            except Exception as e:
                print(f'Could not load checkpoint {path}: {e}')
                continue
    
    if json_path and Path(json_path).exists():
        with open(json_path, 'r') as f:
            data = json.load(f)
            if 'metrics' in data:
                metrics = data['metrics']
            elif 'history' in data:
                metrics = data['history']
            else:
                metrics = data
        print(f'Loaded metrics from JSON: {json_path}')
        return metrics
    
    # Try loading from TensorBoard logs
    try:
        from tensorboard.backend.event_processing import event_accumulator
        log_dir = Path('../../logs/pcam')
        if log_dir.exists():
            ea = event_accumulator.EventAccumulator(str(log_dir))
            ea.Reload()
            
            # Extract scalars
            if 'train_loss' in ea.Tags()['scalars']:
                metrics['train_loss'] = [s.value for s in ea.Scalars('train_loss')]
            if 'val_loss' in ea.Tags()['scalars']:
                metrics['val_loss'] = [s.value for s in ea.Scalars('val_loss')]
            if 'train_accuracy' in ea.Tags()['scalars']:
                metrics['train_acc'] = [s.value for s in ea.Scalars('train_accuracy')]
            if 'val_accuracy' in ea.Tags()['scalars']:
                metrics['val_acc'] = [s.value for s in ea.Scalars('val_accuracy')]
            
            if any(len(v) > 0 for v in metrics.values()):
                print(f'Loaded metrics from TensorBoard logs: {log_dir}')
                return metrics
    except Exception as e:
        print(f'Could not load TensorBoard logs: {e}')
    
    print('No metrics file found. Using sample data for demonstration.')
    # Generate sample data for demonstration
    epochs = np.arange(1, 21)
    metrics = {
        'train_loss': 0.7 * np.exp(-0.1 * epochs) + np.random.normal(0, 0.02, 20),
        'val_loss': 0.75 * np.exp(-0.1 * epochs) + np.random.normal(0, 0.03, 20),
        'train_acc': 1 - 0.3 * np.exp(-0.15 * epochs) + np.random.normal(0, 0.01, 20),
        'val_acc': 1 - 0.35 * np.exp(-0.15 * epochs) + np.random.normal(0, 0.015, 20)
    }
    return metrics

# Try to load metrics
METRICS_PATH = Path('../../results/pcam/metrics.json')
metrics = load_metrics(json_path=METRICS_PATH if METRICS_PATH.exists() else None)
print(f'Metrics loaded: {list(metrics.keys())}')

In [ ]:
# Plot loss curves (train/val) over epochs
fig, ax = plt.subplots(figsize=(12, 6))

epochs = range(1, len(metrics['train_loss']) + 1)

ax.plot(epochs, metrics['train_loss'], 'b-', label='Train Loss', linewidth=2, marker='o', markersize=4)
ax.plot(epochs, metrics['val_loss'], 'r-', label='Val Loss', linewidth=2, marker='s', markersize=4)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training and Validation Loss Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'loss_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved loss curves to {OUTPUT_DIR / "loss_curves.png"}')

In [ ]:
# Plot accuracy curves (train/val) over epochs
fig, ax = plt.subplots(figsize=(12, 6))

epochs = range(1, len(metrics['train_acc']) + 1)

ax.plot(epochs, metrics['train_acc'], 'b-', label='Train Accuracy', linewidth=2, marker='o', markersize=4)
ax.plot(epochs, metrics['val_acc'], 'r-', label='Val Accuracy', linewidth=2, marker='s', markersize=4)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Training and Validation Accuracy Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Set y-axis limits to [0, 1]
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'accuracy_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved accuracy curves to {OUTPUT_DIR / "accuracy_curves.png"}')

## 3. Model Performance

Load and plot confusion matrix, ROC curve with AUC, and precision-recall curve.

In [ ]:
# Load evaluation results
def load_evaluation_results(json_path=None):
    """Load evaluation metrics from JSON."""
    default_results = {
        'confusion_matrix': np.array([[4500, 500], [300, 4700]]),
        'y_true': np.concatenate([np.zeros(4800), np.ones(5000)]),
        'y_pred': np.concatenate([np.zeros(4500) + np.random.normal(0, 0.3, 4500), 
                                  np.ones(4700) + np.random.normal(0, 0.3, 4700)]),
        'y_pred_proba': np.concatenate([np.random.beta(2, 5, 4500), np.random.beta(5, 2, 4700)])
    }
    
    # Try multiple possible paths
    possible_paths = [
        json_path,
        Path('../../results/pcam_eval_test/metrics.json'),
        Path('../../results/pcam/metrics.json')
    ]
    
    for path in possible_paths:
        if path and Path(path).exists():
            with open(path, 'r') as f:
                data = json.load(f)
                print(f'Loaded evaluation results from {path}')
                
                # Convert to expected format
                results = {
                    'confusion_matrix': np.array(data['confusion_matrix']),
                    'y_true': np.array(data['labels']),
                    'y_pred': np.array(data['predictions']),
                    'y_pred_proba': np.array(data['probabilities'])
                }
                return results
    
    print('Using sample evaluation data for demonstration.')
    return default_results

EVAL_PATH = Path('../../results/pcam/metrics.json')
eval_results = load_evaluation_results(json_path=EVAL_PATH if EVAL_PATH.exists() else None)

# Extract confusion matrix
cm = eval_results['confusion_matrix']

print(f'Confusion matrix shape: {cm.shape}')
print(f'Total samples: {len(eval_results["y_true"])}')

In [ ]:
# Load and plot confusion matrix heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal (0)', 'Metastatic (1)'],
            yticklabels=['Normal (0)', 'Metastatic (1)'],
            annot_kws={'size': 14})

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved confusion matrix to {OUTPUT_DIR / "confusion_matrix.png"}')

In [ ]:
# Plot ROC curve with AUC
from sklearn.metrics import roc_curve, auc

y_true = eval_results['y_true']
y_pred_proba = eval_results['y_pred_proba']

fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved ROC curve to {OUTPUT_DIR / "roc_curve.png"}')

In [ ]:
# Plot precision-recall curve
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
avg_precision = average_precision_score(y_true, y_pred_proba)

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(recall, precision, 'g-', linewidth=2, label=f'PR Curve (AP = {avg_precision:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'precision_recall_curve.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved precision-recall curve to {OUTPUT_DIR / "precision_recall_curve.png"}')

## 4. Prediction Analysis

Analyze model predictions by displaying correct/incorrect predictions with confidence scores and plotting confidence histograms.

In [ ]:
# Prepare prediction data for analysis
y_true = eval_results['y_true']
y_pred_proba = eval_results['y_pred_proba']

# Convert probabilities to binary predictions
y_pred = (y_pred_proba >= 0.5).astype(int)

# Identify correct and incorrect predictions
correct_mask = (y_pred == y_true)
incorrect_mask = ~correct_mask

correct_confidence = y_pred_proba[correct_mask]
incorrect_confidence = y_pred_proba[incorrect_mask]

correct_pred_indices = np.where(correct_mask)[0]
incorrect_pred_indices = np.where(incorrect_mask)[0]

print(f'Total predictions: {len(y_true)}')
print(f'Correct predictions: {len(correct_pred_indices)} ({100*len(correct_pred_indices)/len(y_true):.2f}%)')
print(f'Incorrect predictions: {len(incorrect_pred_indices)} ({100*len(incorrect_pred_indices)/len(y_true):.2f}%)')

In [ ]:
# Display correct predictions with confidence (sample of 16)
if DATASET_LOADED and len(correct_pred_indices) > 0:
    n_display = min(16, len(correct_pred_indices))
    display_indices = np.random.choice(correct_pred_indices, size=n_display, replace=False)
    
    n_cols = 4
    n_rows = (n_display + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
    fig.suptitle('Correct Predictions with Confidence', fontsize=14, fontweight='bold')
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for i, ax in enumerate(axes.flat):
        if i < n_display:
            idx = display_indices[i]
            sample = dataset[idx % len(dataset)]
            img = sample['image']
            
            if img.max() <= 1.0:
                img_display = img.numpy().transpose(1, 2, 0)
            else:
                img_display = img.numpy().transpose(1, 2, 0) / 255.0
            
            true_label = y_true[idx]
            confidence = y_pred_proba[idx]
            
            ax.imshow(img_display)
            ax.set_title(f'True={true_label}, Conf={confidence:.3f}', fontsize=9)
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'correct_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved correct predictions visualization')

In [ ]:
# Display incorrect predictions with confidence (sample of 16)
if DATASET_LOADED and len(incorrect_pred_indices) > 0:
    n_display = min(16, len(incorrect_pred_indices))
    display_indices = np.random.choice(incorrect_pred_indices, size=n_display, replace=False)
    
    n_cols = 4
    n_rows = (n_display + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
    fig.suptitle('Incorrect Predictions with Confidence', fontsize=14, fontweight='bold')
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for i, ax in enumerate(axes.flat):
        if i < n_display:
            idx = display_indices[i]
            sample = dataset[idx % len(dataset)]
            img = sample['image']
            
            if img.max() <= 1.0:
                img_display = img.numpy().transpose(1, 2, 0)
            else:
                img_display = img.numpy().transpose(1, 2, 0) / 255.0
            
            true_label = y_true[idx]
            pred_label = y_pred[idx]
            confidence = y_pred_proba[idx]
            
            ax.imshow(img_display)
            ax.set_title(f'True={true_label}, Pred={pred_label}\nConf={confidence:.3f}', fontsize=9)
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'incorrect_predictions.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved incorrect predictions visualization')

In [ ]:
# Plot confidence histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall confidence distribution
ax1 = axes[0]
ax1.hist(y_pred_proba, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold')
ax1.set_xlabel('Predicted Probability', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Overall Confidence Distribution', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Confidence distribution by correctness
ax2 = axes[1]
ax2.hist(correct_confidence, bins=50, color='green', edgecolor='black', alpha=0.6, label=f'Correct (n={len(correct_confidence)})')
ax2.hist(incorrect_confidence, bins=50, color='red', edgecolor='black', alpha=0.6, label=f'Incorrect (n={len(incorrect_confidence)})')
ax2.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold')
ax2.set_xlabel('Predicted Probability', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Confidence Distribution by Prediction Correctness', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confidence_histogram.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved confidence histogram to {OUTPUT_DIR / "confidence_histogram.png"}')

## 5. Save All Plots

Save all plots to the results/pcam/ directory with consistent naming.

In [ ]:
# List all saved plots
import glob

saved_plots = sorted(OUTPUT_DIR.glob('*.png'))
print(f'All plots saved to {OUTPUT_DIR}:')
for plot in saved_plots:
    print(f'  - {plot.name}')

In [ ]:
# Summary statistics
print('\n' + '='*60)
print('VISUALIZATION SUMMARY')
print('='*60)

if DATASET_LOADED:
    print(f'\nDataset Information:')
    print(f'  - Total samples displayed: {len(samples)}')
    print(f'  - Class distribution: Normal={np.sum(labels==0)}, Metastatic={np.sum(labels==1)}')

print(f'\nTraining Metrics:')
print(f'  - Epochs tracked: {len(metrics.get("train_loss", []))}')
if 'train_loss' in metrics and len(metrics['train_loss']) > 0:
    print(f'  - Final train loss: {metrics["train_loss"][-1]:.4f}')
    print(f'  - Final val loss: {metrics["val_loss"][-1]:.4f}')
    print(f'  - Final train acc: {metrics["train_acc"][-1]:.4f}')
    print(f'  - Final val acc: {metrics["val_acc"][-1]:.4f}')

print(f'\nEvaluation Metrics:')
print(f'  - Total predictions: {len(y_true)}')
print(f'  - Accuracy: {100 * np.sum(y_pred == y_true) / len(y_true):.2f}%')

print(f'\nOutput Directory: {OUTPUT_DIR}')
print(f'Total plots saved: {len(saved_plots)}')